In [8]:
K.<w> = QuadraticField(-1, embedding=CC.0) # CC.0 is the imaginary unit 'i'
OK = K.ring_of_integers()
U = K.unit_group()


print(OK)

Gaussian Integers generated by w in Number Field in w with defining polynomial x^2 + 1 with w = 1*I


In [9]:
def closest_integer(gamma):
    K = gamma.parent()
    OK = K.ring_of_integers()
    
    # 1. Get a basis for the ring of integers
    # This basis is {1, omega}
    basis = OK.basis()
    
    # 2. Embed the lattice and target into R^2
    def embed(element):
        c = CC(element) 
        return vector(RR, [c.real(), c.imag()])

    
    # B is the basis matrix
    B = matrix(RR, [embed(b) for b in basis]).transpose()
    target = embed(gamma)
    
    # Get exact coordinates in the lattice basis
    coords = B.solve_right(target)
    
    # 3. Test all 4 combinations of floor and ceiling for the 2 coordinates
    best_dist = infinity
    best_integer = None
    
    import itertools
    # Defines Nearest Neighbours with integer coefficients (note simple rounding would not work for oblique lattices)
    options = [[floor(c)-1, floor(c), floor(c)+1] for c in coords]
    for p, q in itertools.product(*options):
        candidate = OK([p, q])
        # Calculate Euclidean distance squared in the embedding space
        diff = embed(candidate) - target
        dist_sq = diff.dot_product(diff)
        
        if dist_sq < best_dist:
            best_dist = dist_sq
            best_integer = candidate
            
    return best_integer

In [10]:
units = [1, w, w*w, w*w*w]
matrixelements = [0, 1, w, w*w, w*w*w]

In [11]:
# integers of norm at most 1 
S = [K(0), K(1), w, w*w, w*w*w]
T = [K(1), w, w*w, w*w*w]


mats = []
matsupdate = []
for a in S:
    for b in S:
        for c in T:
           for d in S:
               M = matrix(K, [[a, b], [c, d]])
               if M.det() == 1:
                    theta = closest_integer(K(K(-d).quo_rem(K(c))[0])) # closest integer of quotient
                    alpha = K(theta)*K(a) + K(b) # update entry a
                    gamma = K(theta)*K(c) + K(d) # update entry c
                    N = matrix(K, [[alpha, -a], [gamma, -c]])
                    mats.append(M)                 
                    matsupdate.append(N)

In [12]:
def max_abs_squared_norm(M):
    """
    Return max of the norms of the elements in the 2x2 matrix M. Works because we are in a quadratic field
    """
    return max((x*x.galois_conjugate() for x in M.list()))

normmats = []
for m in mats:
    normmats.append(max_abs_squared_norm(m))

In [13]:
normmatsupdate = []
for m in matsupdate:
    normmatsupdate.append(max_abs_squared_norm(m))

In [14]:
result = list(zip(mats, matsupdate, normmats, normmatsupdate))
 
for i, (M, N, x, y) in enumerate(result, 1):
    print(f"#{i:3d} matrix {list(map(list, M.rows()))} update {list(map(list, N.rows()))} | norms=({x}, {y})")



#  1 matrix [[0, 1], [-1, 0]] update [[1, 0], [0, 1]] | norms=(1, 1)
#  2 matrix [[0, 1], [-1, 1]] update [[1, 0], [0, 1]] | norms=(1, 1)
#  3 matrix [[0, 1], [-1, w]] update [[1, 0], [0, 1]] | norms=(1, 1)
#  4 matrix [[0, 1], [-1, -1]] update [[1, 0], [0, 1]] | norms=(1, 1)
#  5 matrix [[0, 1], [-1, -w]] update [[1, 0], [0, 1]] | norms=(1, 1)
#  6 matrix [[0, w], [w, 0]] update [[w, 0], [0, -w]] | norms=(1, 1)
#  7 matrix [[0, w], [w, 1]] update [[w, 0], [0, -w]] | norms=(1, 1)
#  8 matrix [[0, w], [w, w]] update [[w, 0], [0, -w]] | norms=(1, 1)
#  9 matrix [[0, w], [w, -1]] update [[w, 0], [0, -w]] | norms=(1, 1)
# 10 matrix [[0, w], [w, -w]] update [[w, 0], [0, -w]] | norms=(1, 1)
# 11 matrix [[0, -1], [1, 0]] update [[-1, 0], [0, -1]] | norms=(1, 1)
# 12 matrix [[0, -1], [1, 1]] update [[-1, 0], [0, -1]] | norms=(1, 1)
# 13 matrix [[0, -1], [1, w]] update [[-1, 0], [0, -1]] | norms=(1, 1)
# 14 matrix [[0, -1], [1, -1]] update [[-1, 0], [0, -1]] | norms=(1, 1)
# 15 matrix [[0, -1],